In [ ]:
# Import the necessary packages
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Read in our cafe dataset - remember it has to be loaded into Colab environment
cafe_sales = pd.read_csv("cafe_data.csv")
cafe_sales.head()

# We'll take you through various feature engineering techniques. Whether you use any one of these depends entirely on the dataset being worked with.

# Binning
Binning is the process of transforming continuous, often numerical data, into categorical "bins".

Two main methods for binning are equal-width bins, where the value ranges are the same for each bin, and equal-frequency bins, where each bin has the same number of data points.

**pd.cut()** is the function used to create bins.

In [ ]:
# We will bin "Total Spent"
# First let's find the min and max
print(cafe_sales["Total Spent"].min())
print(cafe_sales["Total Spent"].max())

In [ ]:
# Make our bins and bin labels
bins = [1, 5, 10, 15, 20, 26]
bin_labels = ['1-5', '6-10', '11-15', '16-20', '21+']

# Create new binned variable
cafe_sales["Total Spent Binned"] = pd.cut(cafe_sales["Total Spent"], bins, labels=bin_labels)
cafe_sales.head()

# Encoding
There are two, very useful types of encoding: label encoding and one-hot encoding. Both methods turn categorical data into numerical data.

## Label Encoding
Label encoding gives each unique category its own number.

**Label Encoding is usually only acceptable when the categories have a meaningful ranking, such as "poor", "average", and "good".**

**For other situations, like our variables, label encoding is only acceptable for tree-based models.**

In [ ]:
# Label encode for payment method
# Give our methods their encodings
encodings = {"Cash" : 1, "Credit Card": 2, "Digital Wallet": 3, "Unknown": 4}

# Use .map to "map" our encodings
cafe_sales["Payment Method Encoded"] = cafe_sales["Payment Method"].map(encodings)
cafe_sales.iloc[:, [0,1,2,3,4,5,6,8]].head()

## One-Hot Encoding
One-hot encoding creates a new column for each unique category, and gives it a binary 0 or 1 depending on the original value. It is much more versatile than label encoding, but does introduce a lot of new columns.

**pd.get_dummies()** performs the encoding for us.

In [ ]:
# One-hot encode for payment method
dummies = pd.get_dummies(cafe_sales["Payment Method"], prefix="Payment")

# Concatenate our new columns with the original dataset
cafe_sales = pd.concat([cafe_sales, dummies], axis=1)
cafe_sales.iloc[:, [0,1,2,3,4,5,6,9,10,11,12]].head()

# Feature Splitting
Feature Splitting is exactly what it sounds like: splitting a single feature into multiple, more specific features. Transaction Date can provide a good example of this.

**.split() can split strings by any character(s) you specify**

In [ ]:
cafe_sales["Transaction Date"] = pd.to_datetime(cafe_sales["Transaction Date"])

In [ ]:
# Split Transaction Date into Day, Month, Year, and Weekend/Weekday
cafe_sales["Day"] = cafe_sales["Transaction Date"].dt.day
cafe_sales["Month"] = cafe_sales["Transaction Date"].dt.month
cafe_sales["Year"] = cafe_sales["Transaction Date"].dt.year

cafe_sales["Day of Week"] = cafe_sales["Transaction Date"].dt.dayofweek
cafe_sales["Is Weekend"] = cafe_sales["Day of Week"].isin([5, 6])

cafe_sales.iloc[:, [0,1,2,3,4,5,6,13,14,15,16,17]].head()

# Interaction Features

Interaction features are features created using multiple of the existing features. A simple example is using height and weight variables to create a BMI (Body Mass Index) variable. It's important to have a good reason for combining the variables.

In our case, we'll be creating a categorical variable that we can then one-hot encode.

In [ ]:
# Categorical variable combining Item and Location
cafe_sales["Item_Location"] = cafe_sales["Item"] + "_" + cafe_sales["Location"]
cafe_sales.iloc[:, [0,1,2,3,4,5,6,18]].head()

# Correlation Matrix
A correlation matrix is one of the most helpful tools for feature selection, and it helps you see how correlated the existing features and the features you created are with your target variable.

In [ ]:
# One-hot encode Item, Location, and Item_Location
dummies1 = pd.get_dummies(cafe_sales[["Item", "Location"]])

# Concatenate our new columns with the original dataset
cafe_sales = pd.concat([cafe_sales, dummies1], axis=1)

# Drop Payment Method Encoded since its useless
cafe_sales.drop("Payment Method Encoded", axis=1, inplace=True)

In [ ]:
# Create our correlation matrix
corr = cafe_sales.select_dtypes(include=["number", "bool"]).corr()


plt.figure(figsize=(12, 10))

sns.heatmap(
    corr,
    annot=False,
    cmap="coolwarm",
    center=0
)

plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()